# Tool Error Handling and Retries

Tools fail. APIs time out, databases go offline, inputs are malformed. An agent that treats every `ToolResult.error` as fatal will halt at the first hiccup. Robust error handling means the agent retries transient failures, falls back to alternatives, and degrades gracefully when nothing works.

**What you'll build:** retry wrappers, fallback tool chains, and a graceful-degradation agent that continues making progress even when individual tools fail.

**Time:** ~15 min | **Difficulty:** Intermediate

**What you'll learn:**
- How `ToolResult.is_error` signals failure to the agent
- Building a `RetryTool` wrapper with configurable backoff
- Building a `FallbackTool` that tries alternatives in order
- How the agent reasons about tool errors and whether to retry
- Logging errors without disrupting the agent loop

## Prerequisites

You need:
- An **OpenAI API key** (`OPENAI_API_KEY`)
- The `synapsekit` package (installed below)

In [ ]:
!pip install synapsekit -q

In [ ]:
import os

# Set your API key here
# os.environ['OPENAI_API_KEY'] = 'sk-...'

## Step 1: Understand ToolResult error semantics

`ToolResult.is_error` is `True` when the `error` field is not `None`. The `__str__` method returns `error` when set, so the agent always receives a string observation — it never sees `None`.

The agent receives the `str(result)` as its observation. For errors, this means the LLM sees the error message and can decide whether to retry, try a different tool, or report the failure.

In [ ]:
import asyncio
from synapsekit.agents import BaseTool, ToolResult

# Success
ok = ToolResult(output="42")
print(f"is_error: {ok.is_error}")    # False
print(f"str():    {str(ok)}")        # "42"

print()

# Error
err = ToolResult(output="", error="Connection timeout after 5s")
print(f"is_error: {err.is_error}")   # True
print(f"str():    {str(err)}")       # "Connection timeout after 5s"

## Step 2: Build a RetryTool wrapper

A retry wrapper transparently re-calls the underlying tool when it returns an error. Use exponential backoff to avoid hammering a rate-limited API.

In [ ]:
import asyncio
from typing import Any


class RetryTool(BaseTool):
    """Wraps any BaseTool with automatic retry on error."""

    def __init__(
        self,
        inner: BaseTool,
        max_retries: int = 3,
        initial_delay: float = 1.0,
        backoff_factor: float = 2.0,
    ) -> None:
        self._inner = inner
        self._max_retries = max_retries
        self._initial_delay = initial_delay
        self._backoff_factor = backoff_factor

    @property
    def name(self) -> str:
        return self._inner.name

    @property
    def description(self) -> str:
        return self._inner.description

    @property
    def parameters(self) -> dict:
        return self._inner.parameters

    def schema(self) -> dict:
        return self._inner.schema()

    async def run(self, **kwargs: Any) -> ToolResult:
        delay = self._initial_delay

        for attempt in range(self._max_retries + 1):
            result = await self._inner.run(**kwargs)

            if not result.is_error:
                return result  # Success — no retry needed

            if attempt < self._max_retries:
                print(f"[retry] {self.name} failed (attempt {attempt + 1}): {result.error}")
                await asyncio.sleep(delay)
                delay *= self._backoff_factor

        return result  # Return the last error after all retries exhausted

print("RetryTool class defined.")

## Step 3: Build a FallbackTool chain

A fallback chain tries each tool in order and returns the first success. This is ideal when you have a premium tool (e.g., Tavily) and a free fallback (e.g., DuckDuckGo).

In [ ]:
class FallbackTool(BaseTool):
    """Try each tool in order; return the first successful result."""

    def __init__(self, tools: list[BaseTool], name: str, description: str) -> None:
        if not tools:
            raise ValueError("FallbackTool requires at least one tool")
        self._tools = tools
        self._name = name
        self._description = description
        self._parameters = tools[0].parameters

    @property
    def name(self) -> str:
        return self._name

    @property
    def description(self) -> str:
        return self._description

    @property
    def parameters(self) -> dict:
        return self._parameters

    async def run(self, **kwargs: Any) -> ToolResult:
        errors = []

        for tool in self._tools:
            result = await tool.run(**kwargs)
            if not result.is_error:
                return result  # First success wins
            errors.append(f"{tool.name}: {result.error}")

        # All tools failed — return a combined error message
        combined = "; ".join(errors)
        return ToolResult(output="", error=f"All fallbacks failed: {combined}")

print("FallbackTool class defined.")

## Step 4: Build a flaky tool for testing

Always test your error handling logic with a deliberately unreliable tool.

In [ ]:
import random


class FlakeySearchTool(BaseTool):
    """Simulates a search tool that fails randomly — for testing retry logic."""

    name = "search"
    description = "Search the web for information."
    parameters = {
        "type": "object",
        "properties": {"query": {"type": "string", "description": "Search query"}},
        "required": ["query"],
    }

    def __init__(self, failure_rate: float = 0.6) -> None:
        self._failure_rate = failure_rate
        self._call_count = 0

    async def run(self, query: str = "", **kwargs: Any) -> ToolResult:
        self._call_count += 1
        if random.random() < self._failure_rate:
            return ToolResult(output="", error=f"Rate limit exceeded (call #{self._call_count})")
        return ToolResult(output=f"Search results for '{query}': [mock result #{self._call_count}]")


# Test the retry wrapper
random.seed(42)
flakey = FlakeySearchTool(failure_rate=0.7)
reliable = RetryTool(flakey, max_retries=3, initial_delay=0.1)

result = asyncio.run(reliable.run(query="test query"))
status = "ERROR" if result.is_error else "OK"
print(f"[{status}] {str(result)[:80]}")
print(f"Total underlying calls: {flakey._call_count}")

## Step 5: Add error logging without disrupting the agent

Wrap `run()` to log errors to a side channel while still returning the `ToolResult` to the agent.

In [ ]:
class LoggingTool(BaseTool):
    """Wraps a tool to log all errors to a collector."""

    def __init__(self, inner: BaseTool) -> None:
        self._inner = inner
        self.errors: list[dict] = []

    @property
    def name(self) -> str:
        return self._inner.name

    @property
    def description(self) -> str:
        return self._inner.description

    @property
    def parameters(self) -> dict:
        return self._inner.parameters

    async def run(self, **kwargs: Any) -> ToolResult:
        result = await self._inner.run(**kwargs)
        if result.is_error:
            self.errors.append({"tool": self.name, "error": result.error, "kwargs": kwargs})
        return result  # Always forward the result — logging never blocks the agent


# Test the logging wrapper
random.seed(99)
logged_tool = LoggingTool(FlakeySearchTool(failure_rate=0.8))

for q in ["query1", "query2", "query3"]:
    asyncio.run(logged_tool.run(query=q))

print(f"Errors logged: {len(logged_tool.errors)}")
for e in logged_tool.errors:
    print(f"  {e['tool']}: {e['error']}")

## Step 6: Steer the agent to handle errors in system prompt

The agent's behavior when it receives a tool error depends on its system prompt. Be explicit about what the agent should do when a tool fails.

In [ ]:
from synapsekit.agents import FunctionCallingAgent
from synapsekit.llms.openai import OpenAILLM

random.seed(42)
agent = FunctionCallingAgent(
    llm=OpenAILLM(model="gpt-4o-mini"),
    tools=[RetryTool(FlakeySearchTool(failure_rate=0.5), max_retries=2, initial_delay=0.1)],
    system_prompt=(
        "You are a research assistant. "
        "If a tool returns an error message, try calling it again once with the same or simplified input. "
        "If it fails twice in a row, inform the user that the service is temporarily unavailable "
        "and provide the best answer you can from your training knowledge."
    ),
    max_iterations=8,
)

answer = asyncio.run(agent.run("What is the capital of France?"))
print(f"Answer: {answer}")

## Complete Working Example

A self-contained script demonstrating all three scenarios: retry wrapper, fallback chain, and a full agent with error recovery.

In [ ]:
import asyncio
import random
from typing import Any
from synapsekit.agents import BaseTool, FunctionCallingAgent, ToolResult
from synapsekit.llms.openai import OpenAILLM


class FlakeySearchTool(BaseTool):
    name = "web_search"
    description = "Search the web. May occasionally fail due to rate limits."
    parameters = {
        "type": "object",
        "properties": {"query": {"type": "string"}},
        "required": ["query"],
    }

    def __init__(self, failure_rate: float = 0.5) -> None:
        self._failure_rate = failure_rate
        self._calls = 0

    async def run(self, query: str = "", **kwargs: Any) -> ToolResult:
        self._calls += 1
        if random.random() < self._failure_rate:
            return ToolResult(output="", error="503 Service Unavailable")
        return ToolResult(output=f"Results for '{query}': [article 1, article 2, article 3]")


class RetryTool(BaseTool):
    def __init__(self, inner: BaseTool, max_retries: int = 2, delay: float = 0.5) -> None:
        self._inner = inner
        self._max_retries = max_retries
        self._delay = delay

    @property
    def name(self) -> str: return self._inner.name
    @property
    def description(self) -> str: return self._inner.description
    @property
    def parameters(self) -> dict: return self._inner.parameters

    async def run(self, **kwargs: Any) -> ToolResult:
        for attempt in range(self._max_retries + 1):
            result = await self._inner.run(**kwargs)
            if not result.is_error:
                return result
            if attempt < self._max_retries:
                print(f"  [retry {attempt + 1}/{self._max_retries}] {result.error}")
                await asyncio.sleep(self._delay)
        return result


class FallbackTool(BaseTool):
    def __init__(self, tools: list[BaseTool], name: str, description: str) -> None:
        self._tools = tools
        self._name = name
        self._description = description

    @property
    def name(self) -> str: return self._name
    @property
    def description(self) -> str: return self._description
    @property
    def parameters(self) -> dict: return self._tools[0].parameters

    async def run(self, **kwargs: Any) -> ToolResult:
        errors = []
        for t in self._tools:
            result = await t.run(**kwargs)
            if not result.is_error:
                return result
            errors.append(f"{t.name}: {result.error}")
        return ToolResult(output="", error="All options failed: " + "; ".join(errors))


async def main() -> None:
    random.seed(42)

    # Scenario 1: retry wrapper around a flaky tool
    print("=== Scenario 1: Retry wrapper ===")
    flakey = FlakeySearchTool(failure_rate=0.6)
    reliable = RetryTool(flakey, max_retries=3, delay=0.1)

    for q in ["quantum computing", "Python 3.13 features", "SynapseKit release"]:
        result = await reliable.run(query=q)
        status = "ERROR" if result.is_error else "OK"
        print(f"  [{status}] {q}: {str(result)[:70]}")

    print(f"\n  Total underlying calls: {flakey._calls}")

    # Scenario 2: fallback chain
    print("\n=== Scenario 2: Fallback chain ===")
    always_fails = FlakeySearchTool(failure_rate=1.0)  # always fails
    always_fails.name = "premium_search"
    backup = FlakeySearchTool(failure_rate=0.0)        # always succeeds
    backup.name = "fallback_search"

    fallback = FallbackTool(
        tools=[always_fails, backup],
        name="search",
        description="Search with automatic fallback.",
    )
    result = await fallback.run(query="machine learning news")
    print(f"  Result: {result.output}")

    # Scenario 3: full agent with error handling
    print("\n=== Scenario 3: Agent with error recovery ===")
    agent = FunctionCallingAgent(
        llm=OpenAILLM(model="gpt-4o-mini"),
        tools=[RetryTool(FlakeySearchTool(failure_rate=0.4), max_retries=2, delay=0.1)],
        system_prompt=(
            "You are a helpful assistant. If a tool returns an error, "
            "retry once. If it still fails, answer from your knowledge."
        ),
        max_iterations=6,
    )
    answer = await agent.run("What is the capital of Japan?")
    print(f"  Answer: {answer}")


asyncio.run(main())

## Next Steps

- [Multi-Tool Orchestration](https://synapsekit.github.io/synapsekit-docs/docs/guides/agents/multi-tool-orchestration) — apply retry and fallback wrappers to a diverse toolset
- [Agent with Safety Guardrails](https://synapsekit.github.io/synapsekit-docs/docs/guides/agents/agent-with-guardrails) — combine error handling with input/output validation
- [Creating Custom Tools](https://synapsekit.github.io/synapsekit-docs/docs/guides/agents/custom-tool-creation) — build error-resilient tools from scratch with explicit validation